# 📰 실습 4 : 한글 TF-IDF — 뉴스 기사 검색 시스템
> **핵심 개념** : 포털 사이트의 뉴스 검색 기능이 어떻게 동작하는지 TF-IDF로 직접 구현합니다.
>
> | 개념 | 의미 | 예시 |
> |---|---|---|
> | **TF** | 이 기사에서 단어가 얼마나 자주 나왔나 | '전기차'가 5번 등장 |
> | **IDF** | 전체 기사 중 몇 개에 등장하나 (희귀할수록 높음) | '전기차'는 2개 기사에만 등장 |
> | **TF-IDF** | TF × IDF → 이 기사에서만 특별히 중요한 단어 | 해당 기사 핵심 키워드 |

In [ ]:
# STEP 0 : 패키지 설치
!pip install gradio konlpy -q
print("✅ 설치 완료")

In [ ]:
# STEP 1 : 라이브러리 불러오기
import pandas as pd
import numpy as np
import math
import gradio as gr
from konlpy.tag import Okt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("✅ 라이브러리 로드 완료")

In [ ]:
# STEP 2 : 형태소 분석기 준비
okt = Okt()

STOPWORDS = [
    '이','그','저','것','수','등','및','를','을','은','는','이','가','의','에','도',
    '로','으로','하다','있다','되다','이다','없다','않다','하는','있는','되는',
    '이런','그런','저런','위해','통해','대한','관련','따른','이후','현재','대해','지난','오는'
]

def tokenize(text):
    """명사 + 형용사만 추출 (뉴스 키워드 분석에 적합)"""
    tokens = okt.pos(text, norm=True, stem=True)
    result = [
        word for word, pos in tokens
        if pos in ('Noun', 'Adjective')
        and word not in STOPWORDS
        and len(word) > 1
    ]
    return ' '.join(result)

# 테스트
test = "삼성전자가 최신 인공지능 반도체를 개발했습니다"
print(f"원문   : {test}")
print(f"분석 후: {tokenize(test)}")

In [ ]:
# STEP 3 : 뉴스 기사 데이터 준비
ARTICLES_RAW = [
    ("기사1", "IT",   "삼성전자가 최신 인공지능 스마트폰을 출시했습니다. 새로운 AI 카메라 기능이 탑재되어 사진 품질이 크게 향상되었습니다."),
    ("기사2", "IT",   "애플이 새로운 아이폰을 발표했습니다. 향상된 배터리 성능과 강력한 프로세서를 탑재한 것이 특징입니다."),
    ("기사3", "IT",   "네이버가 한국어 대화형 AI 챗봇을 개발했습니다. 자연어 처리 기술을 활용해 사용자 질문에 정확하게 답변합니다."),
    ("기사4", "자동차","현대자동차가 장거리 전기차를 공개했습니다. 1회 충전으로 500km 이상 주행이 가능한 배터리 기술이 적용되었습니다."),
    ("기사5", "자동차","기아자동차가 새로운 전기차 모델을 출시했습니다. 빠른 충전 속도와 넓은 실내 공간이 장점입니다."),
    ("기사6", "스포츠","한국 야구 대표팀이 국제 대회에서 우승했습니다. 선수들의 뛰어난 실력과 팀워크가 빛났습니다."),
    ("기사7", "스포츠","손흥민 선수가 프리미어리그에서 멀티골을 기록했습니다. 팀의 승리를 이끄는 결정적인 골이었습니다."),
    ("기사8", "IT",   "삼성전자가 차세대 인공지능 반도체를 개발했습니다. AI 연산 성능이 기존 대비 3배 향상되었습니다."),
]

LABELS    = [l for l, _, _   in ARTICLES_RAW]
CATS      = [c for _, c, _   in ARTICLES_RAW]
TEXTS_RAW = [t for _, _, t   in ARTICLES_RAW]
TEXTS_TOK = [tokenize(t)     for t in TEXTS_RAW]

print("📰 기사 목록 및 형태소 분석 결과")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
for label, cat, raw, tok in zip(LABELS, CATS, TEXTS_RAW, TEXTS_TOK):
    print(f"[{label}] [{cat}] {raw[:40]}...")
    print(f"  → 토큰: {tok}")
    print()

In [ ]:
# STEP 4 : TF-IDF 원리를 손으로 계산해보기
print("🧮 TF-IDF 직접 계산 — '전기차' 단어, 기사4 대상")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

target_word = "전기차"
target_tok  = TEXTS_TOK[3]   # 기사4 토큰
word_list   = target_tok.split()

# TF
tf = word_list.count(target_word) / len(word_list)
print(f"TF  = '{target_word}' 등장횟수({word_list.count(target_word)}) / 전체단어수({len(word_list)}) = {tf:.4f}")

# IDF
total_docs    = len(TEXTS_TOK)
docs_with_w   = sum(1 for tok in TEXTS_TOK if target_word in tok.split())
idf = math.log(total_docs / docs_with_w)
print(f"IDF = log(전체기사수({total_docs}) / '{target_word}'포함기사수({docs_with_w})) = {idf:.4f}")

# TF-IDF
print(f"TF-IDF = {tf:.4f} × {idf:.4f} = {tf*idf:.4f}")
print()
print("💡 '전기차'는 소수 기사에만 등장 → IDF가 높아 → TF-IDF도 높습니다.")

In [ ]:
# STEP 5 : sklearn TF-IDF 행렬 생성 및 확인
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(TEXTS_TOK)

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=LABELS
)

print("✅ TF-IDF 행렬 생성 완료")
print(f"   → {tfidf_df.shape[0]}개 기사 × {tfidf_df.shape[1]}개 단어")
print()
print("📌 기사별 TF-IDF 핵심 키워드 TOP 3")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
for label, cat in zip(LABELS, CATS):
    row  = tfidf_df.loc[label]
    top3 = row[row > 0].sort_values(ascending=False).head(3)
    kws  = "  |  ".join([f"{w}({v:.3f})" for w, v in top3.items()])
    print(f"  [{label}][{cat}]  {kws}")

In [ ]:
# STEP 6 : 기사 간 유사도 분석
sim_matrix = cosine_similarity(tfidf_matrix)
sim_df = pd.DataFrame(sim_matrix, index=LABELS, columns=LABELS)

print("📊 기사 간 코사인 유사도 행렬")
print("값이 클수록 내용이 비슷한 기사입니다.")
print()
print(sim_df.round(3).to_string())
print()
print("💡 같은 카테고리(IT·자동차·스포츠) 기사끼리 유사도가 높게 나타납니다.")

In [ ]:
# STEP 7 : Gradio 앱 — 한국어 뉴스 검색 시스템

FULL_LABELS = [f"{l}[{c}]: {t[:30]}..." for l, c, t in zip(LABELS, CATS, TEXTS_RAW)]

# ─ 탭1: 형태소 분석 확인 ─
def show_morphs(article_label):
    idx    = FULL_LABELS.index(article_label)
    tokens = okt.pos(TEXTS_RAW[idx], norm=True, stem=True)
    nouns  = [w for w,p in tokens if p=='Noun'      and len(w)>1]
    adjs   = [w for w,p in tokens if p=='Adjective' and len(w)>1]
    final  = [w for w,p in tokens if p in ('Noun','Adjective')
                                  and w not in STOPWORDS and len(w)>1]
    result  = f"🔬 [{article_label}] 형태소 분석\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
    result += f"원문 : {TEXTS_RAW[idx]}\n\n"
    result += f"명사  ({len(nouns):2d}개) : {' | '.join(nouns)}\n"
    result += f"형용사({len(adjs):2d}개) : {' | '.join(adjs)}\n"
    result += f"\n✅ 불용어 제거 후 최종 ({len(final)}개) :\n  " + "  |  ".join(final)
    return result

# ─ 탭2: 기사 핵심 키워드 ─
def show_keywords(article_label, top_n):
    idx = FULL_LABELS.index(article_label)
    row = tfidf_df.iloc[idx]
    top = row[row > 0].sort_values(ascending=False).head(int(top_n))
    result  = f"🔑 [{article_label}]\n핵심 키워드 TOP {int(top_n)} (TF-IDF 점수)\n"
    result += "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
    for word, score in top.items():
        bar = "█" * int(score * 30)
        result += f"  {word:<12} {score:.4f}  {bar}\n"
    result += "\n💡 TF-IDF가 높을수록 이 기사에서만 특별히 중요한 단어입니다."
    return result

# ─ 탭3: 유사 기사 추천 ─
def find_similar(article_label, top_n):
    idx    = FULL_LABELS.index(article_label)
    scores = sorted(enumerate(cosine_similarity(tfidf_matrix)[idx]),
                    key=lambda x: x[1], reverse=True)
    result = f"🔍 [{article_label}]\n유사 기사 TOP {int(top_n)}\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n"
    filtered = [(i,s) for i,s in scores if i != idx][:int(top_n)]
    for rank, (i, score) in enumerate(filtered, 1):
        bar = "█" * int(score * 25)
        result += f"[{rank}위]  유사도: {score:.3f}  {bar}\n"
        result += f"  → {FULL_LABELS[i]}\n\n"
    return result

# ─ 탭4: 한국어 뉴스 검색 ─
def search_news(query):
    if not query.strip():
        return "검색어를 입력해주세요."
    q_tok = tokenize(query)
    if not q_tok.strip():
        return "유효한 단어가 없습니다. 명사나 형용사를 포함해서 입력해주세요."
    all_tok  = TEXTS_TOK + [q_tok]
    tv_tmp   = TfidfVectorizer()
    mat_tmp  = tv_tmp.fit_transform(all_tok)
    scores   = cosine_similarity(mat_tmp[-1], mat_tmp[:-1])[0]
    top3     = np.argsort(scores)[::-1][:3]
    result   = f"🔍 검색어  : '{query}'\n"
    result  += f"   형태소  : '{q_tok}'\n"
    result  += "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n"
    for rank, idx in enumerate(top3, 1):
        score = scores[idx]
        bar   = "█" * int(score * 25) if score > 0 else "관련도 낮음"
        result += f"[{rank}위]  유사도: {score:.3f}  {bar}\n"
        result += f"  {FULL_LABELS[idx]}\n\n"
    if max(scores) == 0:
        result += "⚠️ 관련 기사를 찾지 못했습니다. 다른 키워드로 시도해보세요."
    return result

with gr.Blocks(theme=gr.themes.Soft(), title="한글 TF-IDF 실습") as app:
    gr.Markdown("# 📰 한글 TF-IDF 실습 — 뉴스 기사 검색 시스템")
    gr.Markdown("> **KoNLPy 형태소 분석** + **TF-IDF** 로 실제 뉴스 검색 엔진 원리를 구현합니다.")

    with gr.Tab("🔬 형태소 분석 확인"):
        gr.Markdown("### 기사를 선택하면 형태소 분석 과정을 보여줍니다.")
        art1 = gr.Dropdown(choices=FULL_LABELS, label="기사 선택", value=FULL_LABELS[0])
        btn1 = gr.Button("분석하기", variant="primary")
        out1 = gr.Textbox(label="형태소 분석 결과", lines=10)
        btn1.click(show_morphs, inputs=art1, outputs=out1)

    with gr.Tab("🔑 핵심 키워드 추출"):
        gr.Markdown("### 기사의 TF-IDF 점수가 높은 핵심 단어를 확인합니다.")
        art2   = gr.Dropdown(choices=FULL_LABELS, label="기사 선택", value=FULL_LABELS[0])
        topn2  = gr.Slider(minimum=3, maximum=10, value=5, step=1, label="상위 N개")
        btn2   = gr.Button("키워드 추출", variant="primary")
        out2   = gr.Textbox(label="핵심 키워드", lines=12)
        btn2.click(show_keywords, inputs=[art2, topn2], outputs=out2)

    with gr.Tab("🔗 유사 기사 추천"):
        gr.Markdown("### 기사를 선택하면 내용이 유사한 기사를 추천합니다.")
        art3   = gr.Dropdown(choices=FULL_LABELS, label="기준 기사", value=FULL_LABELS[0])
        topn3  = gr.Slider(minimum=1, maximum=5, value=3, step=1, label="추천 기사 수")
        btn3   = gr.Button("유사 기사 찾기", variant="primary")
        out3   = gr.Textbox(label="추천 결과", lines=14)
        btn3.click(find_similar, inputs=[art3, topn3], outputs=out3)

    with gr.Tab("🔍 뉴스 검색"):
        gr.Markdown("### 한국어 키워드를 입력하면 관련 기사를 검색합니다.")
        gr.Markdown("예시 검색어: `전기차 배터리`, `인공지능 반도체`, `야구 우승`, `스마트폰 카메라`")
        query_in = gr.Textbox(label="검색어 입력", value="전기차 배터리 충전", lines=1)
        btn4     = gr.Button("검색", variant="primary")
        out4     = gr.Textbox(label="검색 결과", lines=14)
        btn4.click(search_news, inputs=query_in, outputs=out4)

app.launch(share=True)